In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2

In [2]:
IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 50

In [3]:
def load_image_safe(path, size=IMG_SIZE):
    def _load(path_tensor):
        # Convert EagerTensor to string path
        path_str = path_tensor.numpy().decode("utf-8")
        img = Image.open(path_str).convert("RGB")
        img = img.resize((size, size))
        img = np.array(img).astype(np.float32) / 255.0
        return img
    # tf.py_function wraps a Python function that expects numpy inputs
    img = tf.py_function(_load, [path], Tout=tf.float32)
    img.set_shape([size, size, 3])
    return img


In [4]:
def create_dataset_safe(real_dir, cartoon_dir, batch_size=BATCH_SIZE):
    real_images = sorted([os.path.join(real_dir, f) 
                          for f in os.listdir(real_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    cartoon_images = sorted([os.path.join(cartoon_dir, f)
                             for f in os.listdir(cartoon_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(real_images) != len(cartoon_images):
        raise ValueError("Number of real and cartoon images must match!")
    real_ds = tf.data.Dataset.from_tensor_slices(real_images).map(lambda x: load_image_safe(x))
    cartoon_ds = tf.data.Dataset.from_tensor_slices(cartoon_images).map(lambda x: load_image_safe(x))
    ds = tf.data.Dataset.zip((real_ds, cartoon_ds))
    ds = ds.shuffle(100).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [5]:
train_ds = create_dataset_safe("dataset/train/real", "dataset/train/cartoon")
val_ds = create_dataset_safe("dataset/val/real", "dataset/val/cartoon")

In [82]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    return x

def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2,2))(c)
    return c, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(shape=input_shape)
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)
    b = conv_block(p4, 1024)
    d1 = decoder_block(b, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    outputs = layers.Conv2D(3, 1, activation='sigmoid')(d4)
    model = models.Model(inputs, outputs)
    return model

model = build_unet()
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_95 (Conv2D)  │ (None, 128, 128,  │      1,792 │ input_layer_5[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_96 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_95[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_20    │ (None, 64, 64,    │          0 │ conv2d_96[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_97 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_20… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_98 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_97[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_21    │ (None, 32, 32,    │          0 │ conv2d_98[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_99 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_21… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_100 (Conv2D) │ (None, 32, 32,    │    590,080 │ conv2d_99[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_22    │ (None, 16, 16,    │          0 │ conv2d_100[0][0]  │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_101 (Conv2D) │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_22… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_102 (Conv2D) │ (None, 16, 16,    │  2,359,808 │ conv2d_101[0][0]  │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_23    │ (None, 8, 8, 512) │          0 │ conv2d_102[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_103 (Conv2D) │ (None, 8, 8,      │  4,719,616 │ max_pooling2d_23… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_104 (Conv2D) │ (None, 8, 8,      │  9,438,208 │ conv2d_103[0][0]  │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_20 │ (None, 16, 16,    │  2,097,664 │ conv2d_104[0][0]  │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_20      │ (None, 16, 16,    │          0 │ conv2d_transpose

 Total params: 31,031,875 (118.38 MB)

 Trainable params: 31,031,875 (118.38 MB)

 Non-trainable params: 0 (0.00 B)

In [83]:
def edge_loss(y_true, y_pred):
    sobel_true = tf.image.sobel_edges(y_true)
    sobel_pred = tf.image.sobel_edges(y_pred)
    return tf.reduce_mean(tf.abs(sobel_true - sobel_pred))

In [84]:
def combined_loss(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred)) + 0.1 * edge_loss(y_true, y_pred)

In [85]:
model.compile(optimizer='adam', loss=combined_loss)
checkpoint = tf.keras.callbacks.ModelCheckpoint("cartoonmmm.keras", save_best_only=True, monitor='val_loss')

In [86]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint]
)

Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 48s 3s/step - loss: 0.2468 - val_loss: 0.1815
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - loss: 0.2231 - val_loss: 0.1693
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - loss: 0.1753 - val_loss: 0.1525
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 46s 3s/step - loss: 0.1596 - val_loss: 0.1420
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 43s 3s/step - loss: 0.1553 - val_loss: 0.1549
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - loss: 0.1569 - val_loss: 0.1416
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - loss: 0.1506 - val_loss: 0.1454
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - loss: 0.1508 - val_loss: 0.1431
Epoch 9/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - loss: 0.1493 - val_loss: 0.1458
Epoch 10/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 51s 3s/step - loss: 0.1486 - val_loss: 0.1412
Epoch 11/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - loss: 0.1502 - val_loss: 0.1410
Epoch 12/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 48s 3s/step - loss: 0.1520 - val_lo

In [2]:
from tensorflow.keras.models import load_model

# Define your custom loss exactly as it was during training
def combined_loss(y_true, y_pred):
    # Example: replace with your actual loss function
    import tensorflow as tf
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Load model with custom loss
model = load_model("bestcartoon.keras", custom_objects={'combined_loss': combined_loss})


In [3]:
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array


In [13]:

def cartoonize_postprocess(img, k=8, smooth_strength=0.4, color_preserve=True):
    img = np.array(img * 255, dtype=np.uint8)
    smooth = cv2.bilateralFilter(img, d=9, sigmaColor=90, sigmaSpace=90)

    if not color_preserve:
        Z = img.reshape((-1, 3))
        Z = np.float32(Z)
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
        _, label, center = cv2.kmeans(Z, k, None, criteria, 10, cv2.KMEANS_PP_CENTERS)
        center = np.uint8(center)
        res = center[label.flatten()]
        smooth = res.reshape(img.shape)

    dark = cv2.convertScaleAbs(smooth, alpha=1.2, beta=-25)
    return dark.astype(np.uint8) / 255.0


In [ ]:

def enhance_cartoon_realistic_soft_edges(original, cartoon,
                                         line_thin=1,            # edge thickness
                                         edge_intensity=0.3,     # reduced edge intensity
                                         blur_sigma=0.2,         # slightly smooth edges
                                         color_strength=1.2,     # overall color boost
                                         blend_with_original=0.6 # preserve inside colors
                                         ):
    import cv2, numpy as np, tensorflow as tf

    # --- Convert tensors to numpy ---
    if isinstance(cartoon, tf.Tensor):
        cartoon = cartoon.numpy()
    if isinstance(original, tf.Tensor):
        original = original.numpy()

    cartoon = np.clip(cartoon, 0, 1)
    original = np.clip(original, 0, 1)

    # --- Step 1: Blend cartoon with original colors for realistic tones ---
    cartoon_colored = np.clip(cartoon * (1 - blend_with_original) + original * blend_with_original, 0, 1)

    # --- Step 2: Detect edges ---
    gray = cv2.cvtColor((original * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    # --- Step 3: Dilate edges slightly ---
    kernel = np.ones((max(1, int(line_thin)), max(1, int(line_thin))), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)

    # --- Step 4: Create soft edge mask ---
    edges_mask = 1 - (edges / 255.0)
    edges_mask = 1 - ((1 - edges_mask) * edge_intensity)  # soften edges
    if blur_sigma > 0:
        edges_mask = cv2.GaussianBlur(edges_mask, (0, 0), blur_sigma)
    edges_mask = np.stack([edges_mask]*3, axis=-1)

    # --- Step 5: Blend edges with cartoon_colored instead of pure black ---
    cartoon_with_edges = cartoon_colored * edges_mask + cartoon_colored * (1 - edges_mask) * 0  # optional minimal effect
    # Or simpler:
    cartoon_with_edges = cartoon_colored * edges_mask  # edges slightly darker than inside

    # --- Step 6: Boost overall colors slightly ---
    cartoon_with_edges = np.clip(cartoon_with_edges * color_strength, 0, 1)

    return cartoon_with_edges


In [ ]:
def cartoonify_image(model, img_path):
    import cv2, numpy as np, matplotlib.pyplot as plt
    from tensorflow.keras.preprocessing.image import load_img

    # --- Load and preprocess ---
    img = load_img(img_path).convert("RGB")
    orig_w, orig_h = img.size
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img_resized)/255.0

    # --- Predict cartoon ---
    input_img = np.expand_dims(img_array, 0)
    cartoon = model.predict(input_img, verbose=0)[0]
    cartoon = cartoonize_postprocess(cartoon)

    # --- Apply realistic colors with soft edges ---

    cartoon = enhance_cartoon_realistic_soft_edges(
    original=img_array,
    cartoon=cartoon,
    line_thin=0.5,        # thinner edges
    edge_intensity=0.2,   # softer edges
    blur_sigma=0.2,       # slightly smooth edges
    color_strength=1.2,   # boost inside colors
    blend_with_original=0.7 # preserve realistic human tones
)
    # Optional minor contrast boost
    cartoon = np.clip(cartoon * 1.1, 0, 1)

    # --- Resize back to original resolution ---
    cartoon = cv2.resize(cartoon, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)

    # --- Display ---
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(np.array(img)/255.0)
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.title("Cartoonified")
    plt.imshow(cartoon)
    plt.axis('off')
    plt.show()

    return cartoon
